In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_26.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_33.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_49.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_44.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_38.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_22.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_30.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_57.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_46.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_58.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_21.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_59.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_32.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_69.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_19.csv
/kaggle/input/datasets/vigneshvenkateswa

# Load

In [4]:
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Preprocess

In [ ]:
import pandas as pd
import numpy as np
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

processed = "/kaggle/input/datasets/amruthanmimatiz/bot-processed/BoT_preprocessed.csv"
output_path = "/kaggle/working/BoT_preprocessed_v2.csv"

sample = pd.read_csv(processed, nrows=500)

dtype_map = {}
for col in sample.select_dtypes('float64').columns:
    dtype_map[col] = np.float32
for col in sample.select_dtypes('int64').columns:
    dtype_map[col] = np.int32

target = 'Attack' if 'Attack' in sample.columns else 'attack'

# Identify column types ONCE from sample (before any encoding)
original_cat_cols = [c for c in sample.select_dtypes(include='object').columns if c != target]
original_num_cols = [c for c in sample.select_dtypes(include=np.number).columns if c != target]

print(f"Categorical cols : {len(original_cat_cols)}")
print(f"Numerical cols   : {len(original_num_cols)}")
print(f"Target           : {target}")

del sample
gc.collect()


Categorical cols : 2
Numerical cols   : 37
Target           : Attack


0

In [4]:
print("Collecting unique values for LabelEncoders...")

unique_vals = {col: set() for col in original_cat_cols + [target]}

for chunk in pd.read_csv(processed, dtype=dtype_map, chunksize=100_000,
                         usecols=original_cat_cols + [target]):
    for col in unique_vals:
        if col in chunk.columns:
            unique_vals[col].update(chunk[col].astype(str).unique())

    del chunk
    gc.collect()

# Fit encoders once on all known values
le_dict = {}
for col in unique_vals:
    le = LabelEncoder()
    le.fit(sorted(unique_vals[col]))   # sorted for reproducibility
    le_dict[col] = le

label_map = dict(zip(le_dict[target].classes_,
                     le_dict[target].transform(le_dict[target].classes_)))
print("Label map:", label_map)
gc.collect()

Label map: {np.str_('Benign'): np.int64(0), np.str_('DDoS'): np.int64(1), np.str_('DoS'): np.int64(2), np.str_('Reconnaissance'): np.int64(3), np.str_('Theft'): np.int64(4)}


0

In [ ]:
first_chunk = True

for chunk in pd.read_csv(processed, dtype=dtype_map, chunksize=100_000):

    # ── Basic cleaning ──────────────────
    drop_cols = [c for c in chunk.columns if chunk[c].nunique() <= 1]
    chunk.drop(columns=drop_cols, inplace=True)
    chunk.drop_duplicates(inplace=True)
    chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
    chunk.dropna(inplace=True)

    if chunk.empty:
        continue

    # ── Encode categorical columns ──────
    for col in original_cat_cols:
        if col not in chunk.columns:
            continue
        chunk[col] = le_dict[col].transform(chunk[col].astype(str)).astype(np.int16)

    # ── Encode target ───────────────────
    chunk[target] = le_dict[target].transform(chunk[target].astype(str)).astype(np.int8)

    # ── Write to disk ───────────────────
    if first_chunk:
        chunk.to_csv(output_path, index=False, mode='w')
        first_chunk = False
    else:
        chunk.to_csv(output_path, index=False, header=False, mode='a')

    del chunk
    gc.collect()

print("All chunks processed and saved to disk.")

In [ ]:
print("Loading sample for heatmap...")

sample_df = pd.read_csv(output_path, nrows=50_000)

num_cols_final = [c for c in sample_df.select_dtypes(include=np.number).columns
                  if c != target]

corr = sample_df[num_cols_final].corrwith(sample_df[target]).abs().sort_values(ascending=False)
top_feats = corr.head(20).index.tolist()

plt.figure(figsize=(12, 8))
sns.heatmap(sample_df[top_feats + [target]].corr(), cmap='coolwarm', annot=False)
plt.title("Top 20 Feature Correlation (sample-based)")
plt.tight_layout()
plt.show()

print("Label map      :", label_map)
print("Output saved to:", output_path)

del sample_df
gc.collect()

# Post Processing

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
import gc
import os
import time
import json
import shutil
from contextlib import contextmanager
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

In [ ]:
DATASET     = 'BoT'
CSV_PATH    = '/kaggle/working/BoT_preprocessed_v2.csv'
SAVE_DIR    = f'/kaggle/working/{DATASET}/'
RESULTS     = f'{SAVE_DIR}results/'
NUM_CLASSES = 5
TARGET_CANDIDATES = ['Attack', 'attack', 'Cat', 'cat']
DROP_COLS   = ['Label', 'label', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Timestamp', 'timestamp']

WINDOW = 10
STEP   = 5
K      = 5
SEEDS  = list(range(1, 21))
BOUNDARY_TOL = 0.005
CLIP_PCT     = (1, 99)
VAR_THRESH   = 0.01
CORR_THRESH  = 0.95
SHAP_CUM     = 0.90
FOLD_SEED    = 42

CHUNKSIZE      = 50_000
FIT_ROWS       = 2_000_000
CORR_ROWS      = 50_000
SHAP_FIT_ROWS  = 100_000
SHAP_EVAL_ROWS = 1_000
SURROGATE_KW   = {}
MAX_SMOTE      = 200_000
ES_FRACTION    = 0.10
ES_MAX         = 50_000

for sub in ['', 'runs', 'timing', 'shap', 'features', 'balance', 'fold_cache',
            'search', 'sensitivity', 'profile', 'stats', 'summary']:
    os.makedirs(f'{RESULTS}{sub}', exist_ok=True)

RESUME_FROM  = None
TIME_LIMIT_H = 10
SESSION_START = time.perf_counter()
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, RESULTS, dirs_exist_ok=True)


def out_of_time():
    return (time.perf_counter() - SESSION_START) / 3600 > TIME_LIMIT_H


def append_csv(path, rows):
    pd.DataFrame(rows).to_csv(path, mode='a', header=not os.path.exists(path), index=False)


@contextmanager
def stage_timer(stage, fold=-1, seed=-1, config='full'):
    start = time.perf_counter()
    yield
    append_csv(f'{RESULTS}timing/stages_{DATASET}.csv',
               [{'stage': stage, 'config': config, 'fold': fold, 'seed': seed,
                 'seconds': time.perf_counter() - start}])

In [ ]:
def clean_columns(df):
    source = next(c for c in TARGET_CANDIDATES if c in df.columns)
    df = df.rename(columns={source: 'target'})
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
    return df.loc[:, ~df.columns.duplicated()]


def stratified_block_edges(y, k, tol, n_classes, grid=201):
    n = len(y)
    radius = int((n // k) * tol)
    global_share = np.bincount(y, minlength=n_classes) / n
    edges = [0]
    for i in range(1, k):
        ideal = i * n // k
        cands = np.unique(np.linspace(ideal - radius, ideal + radius, grid).astype(np.int64))
        counts = np.bincount(y[edges[-1]:cands[0]], minlength=n_classes).astype(np.int64)
        best, best_score = cands[0], None
        for j, cut in enumerate(cands):
            if j:
                counts += np.bincount(y[cands[j - 1]:cut], minlength=n_classes)
            score = (int((counts == 0).sum()),
                     float(np.abs(counts / counts.sum() - global_share).sum()))
            if best_score is None or score < best_score:
                best, best_score = cut, score
        edges.append(int(best))
    edges.append(n)
    return list(zip(edges[:-1], edges[1:]))


with stage_timer('preprocessing'):
    head = clean_columns(pd.read_csv(CSV_PATH, nrows=1000))
    FEATURES = [c for c in head.select_dtypes(include=np.number).columns if c != 'target']

    n_rows, classes = 0, set()
    for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)
        n_rows += len(chunk)
        classes.update(chunk['target'].unique().tolist())

    le = LabelEncoder().fit(sorted(classes))
    assert len(le.classes_) == NUM_CLASSES, f'{len(le.classes_)} classes, expected {NUM_CLASSES}'

    ROWS = np.lib.format.open_memmap(f'{SAVE_DIR}rows.npy', mode='w+', dtype=np.float32,
                                     shape=(n_rows, len(FEATURES)))
    LABELS = np.empty(n_rows, dtype=np.int32)
    pos = 0
    for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNKSIZE, low_memory=False):
        chunk = clean_columns(chunk)
        ROWS[pos:pos + len(chunk)] = chunk[FEATURES].values.astype(np.float32)
        LABELS[pos:pos + len(chunk)] = le.transform(chunk['target'])
        pos += len(chunk)
        del chunk
    ROWS.flush()
    gc.collect()

    BLOCKS = stratified_block_edges(LABELS, K, BOUNDARY_TOL, NUM_CLASSES)

print(f'Rows: {n_rows:,} | Features: {len(FEATURES)} | Classes: {NUM_CLASSES}')
block_rows = []
for b, (a, z) in enumerate(BLOCKS):
    counts = np.bincount(LABELS[a:z], minlength=NUM_CLASSES)
    block_rows.append({'block': b, 'start': a, 'end': z,
                       **{f'class_{c}': int(v) for c, v in enumerate(counts)}})
    if (counts == 0).any():
        print(f'WARNING: block {b} is missing classes {np.flatnonzero(counts == 0).tolist()}')
pd.DataFrame(block_rows).to_csv(f'{RESULTS}features/{DATASET}_blocks.csv', index=False)
print(pd.DataFrame(block_rows)[['block', 'start', 'end']])

In [ ]:
def sample_train_rows(k, n, seed):
    ranges = [BLOCKS[b] for b in range(K) if b != k]
    sizes = np.array([z - a for a, z in ranges])
    total = int(sizes.sum())
    if n is None or n >= total:
        return np.concatenate([np.arange(a, z) for a, z in ranges])
    draws = np.sort(np.random.default_rng(seed).choice(total, n, replace=False))
    offsets = np.concatenate([[0], np.cumsum(sizes)])
    idx = np.empty(n, dtype=np.int64)
    for i, (a, _) in enumerate(ranges):
        mask = (draws >= offsets[i]) & (draws < offsets[i + 1])
        idx[mask] = draws[mask] - offsets[i] + a
    return idx


def scale(x, lo, hi):
    span = hi - lo
    return np.where(span > 0, (np.clip(x, lo, hi) - lo) / np.where(span > 0, span, 1), 0).astype(np.float32)


def mean_abs_shap(values, n_features):
    arr = np.abs(np.stack(values, axis=-1) if isinstance(values, list) else np.asarray(values))
    if arr.ndim == 2:
        return arr.mean(axis=0)
    feature_axis = 1 if arr.shape[1] == n_features else 2
    return arr.mean(axis=tuple(a for a in range(3) if a != feature_axis))


def fit_fold(k):
    cache = f'{RESULTS}fold_cache/fold{k}.npz'
    if os.path.exists(cache):
        saved = np.load(cache)
        return {'lo': saved['lo'], 'hi': saved['hi'], 'fidx': saved['fidx']}

    with stage_timer('preprocessing', k):
        fit_x = ROWS[sample_train_rows(k, FIT_ROWS, FOLD_SEED + k)]
        lo, hi = np.percentile(fit_x, CLIP_PCT, axis=0)

    with stage_timer('variance_correlation', k):
        fit_x = scale(fit_x, lo, hi)
        keep = np.flatnonzero(fit_x.var(axis=0) > VAR_THRESH)
        n_var = len(keep)
        corr_x = fit_x if CORR_ROWS is None else fit_x[::max(1, len(fit_x) // CORR_ROWS)]
        upper = np.triu(pd.DataFrame(corr_x[:, keep]).corr().abs().values, k=1)
        keep = keep[~(upper > CORR_THRESH).any(axis=0)]
    del fit_x, corr_x
    gc.collect()

    with stage_timer('shap', k):
        idx = sample_train_rows(k, SHAP_FIT_ROWS, FOLD_SEED + k + 100)
        X = scale(ROWS[idx][:, keep], lo[keep], hi[keep])
        y = LABELS[idx]
        present = np.bincount(y)
        stratify = y if present[present > 0].min() >= 2 else None
        X_fit, X_held, y_fit, _ = train_test_split(X, y, test_size=0.2, stratify=stratify,
                                                   random_state=FOLD_SEED)
        surrogate = lgb.LGBMClassifier(n_estimators=100, n_jobs=-1, random_state=FOLD_SEED,
                                       verbose=-1, **SURROGATE_KW)
        surrogate.fit(X_fit, y_fit)
        values = shap.TreeExplainer(surrogate).shap_values(X_held[:SHAP_EVAL_ROWS])
        importance = mean_abs_shap(values, len(keep))

        shap_df = pd.DataFrame({'feature': [FEATURES[i] for i in keep], 'col': keep,
                                'mean_abs_shap': importance})
        shap_df = shap_df.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
        shap_df['share'] = shap_df['mean_abs_shap'] / shap_df['mean_abs_shap'].sum()
        shap_df['cumulative'] = shap_df['share'].cumsum()
        top_k = int((shap_df['cumulative'] < SHAP_CUM).sum()) + 1
        shap_df['rank'] = np.arange(1, len(shap_df) + 1)
        shap_df['retained'] = shap_df['rank'] <= top_k
        fidx = shap_df.head(top_k)['col'].values

    shap_df.drop(columns='col').to_csv(f'{RESULTS}shap/{DATASET}_fold{k}.csv', index=False)
    append_csv(f'{RESULTS}features/{DATASET}_cascade.csv',
               [{'fold': k, 'analysis_columns': len(FEATURES), 'after_variance': n_var,
                 'after_correlation': len(keep), 'top_k': top_k}])
    print(f'Fold {k}: {len(FEATURES)} -> {n_var} -> {len(keep)} -> TOP-K {top_k}: '
          f'{shap_df.head(top_k)["feature"].tolist()}')

    np.savez(cache, lo=lo, hi=hi, fidx=fidx)
    del X, y, X_fit, X_held, y_fit, surrogate, values
    gc.collect()
    return {'lo': lo, 'hi': hi, 'fidx': fidx}

In [ ]:
def block_starts(b, W, S):
    a, z = BLOCKS[b]
    if b > 0:
        a += W - 1
    return np.arange(a, z - W + 1, S, dtype=np.int64)


def load_windows(starts, W, art, batch=8192):
    fidx, lo, hi = art['fidx'], art['lo'][art['fidx']], art['hi'][art['fidx']]
    offsets = np.arange(W)
    for i in range(0, len(starts), batch):
        idx = starts[i:i + batch, None] + offsets
        x = ROWS[idx.ravel()][:, fidx]
        yield scale(x, lo, hi).reshape(len(idx), W, len(fidx))


def materialize(starts, W, art):
    return np.concatenate(list(load_windows(starts, W, art)))


def training_windows(k, seed, W, S, art):
    rng = np.random.default_rng(seed * 100 + k)
    tr_parts, es_parts = [], []
    for b in range(K):
        if b != k:
            starts = block_starts(b, W, S)
            cut = int(len(starts) * (1 - ES_FRACTION))
            es_parts.append(starts[cut:])
            tr_parts.append(starts[:cut][starts[:cut] <= starts[cut] - W])
    tr_starts, es_starts = np.concatenate(tr_parts), np.concatenate(es_parts)
    if len(tr_starts) > MAX_SMOTE:
        tr_starts = np.sort(rng.choice(tr_starts, MAX_SMOTE, replace=False))
    if len(es_starts) > ES_MAX:
        es_starts = np.sort(rng.choice(es_starts, ES_MAX, replace=False))
    return (materialize(tr_starts, W, art), LABELS[tr_starts + W - 1],
            materialize(es_starts, W, art), LABELS[es_starts + W - 1])


def balance(X, y, seed, k, config):
    N, W, F = X.shape
    before = np.bincount(y, minlength=NUM_CLASSES)
    if (before == 0).any():
        print(f'WARNING: fold {k} seed {seed} has no training windows for classes '
              f'{np.flatnonzero(before == 0).tolist()}')
    present = np.flatnonzero(before >= 2)
    strategy = {int(c): int(before.max()) for c in present}
    smt = SMOTETomek(smote=SMOTE(sampling_strategy=strategy,
                                 k_neighbors=min(5, int(before[present].min()) - 1),
                                 random_state=seed),
                     tomek=TomekLinks(sampling_strategy='all', n_jobs=-1), random_state=seed)
    X_res, y_res = smt.fit_resample(X.reshape(N, W * F), y)
    after = np.bincount(y_res, minlength=NUM_CLASSES)
    append_csv(f'{RESULTS}balance/{DATASET}.csv',
               [{'config': config, 'fold': k, 'seed': seed, 'class': c,
                 'before': int(before[c]), 'after': int(after[c])} for c in range(NUM_CLASSES)])
    return X_res.reshape(-1, W, F).astype(np.float32), y_res

# Modeling

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.flop_counter import FlopCounterMode
from scipy import stats as scipy_stats
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import lightgbm as lgb
import matplotlib.pyplot as plt
import random
import glob
import gc
import os

In [ ]:
TABLE_I = {
    'channels': [32, 64], 'd_model': 64, 'n_heads': 4, 'lr': 1e-3,
    'batch': 256, 'epochs': 30, 'patience': 7,
    'num_leaves': 63, 'lgb_lr': 0.05, 'lgb_rounds': 30,
}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | Dataset: {DATASET} | Classes: {NUM_CLASSES}')

RUN_MAIN        = True
RUN_SEARCH      = False
USE_SEARCH_BEST = False
RUN_SWEEP       = False
RUN_PROFILE     = True
N_TRIALS        = 30
SWEEP_SEEDS     = SEEDS
SWEEP_W         = [5, 10, 15, 20, 25]
SWEEP_S         = [1, 5, 10]
STATS_DIRS      = [f'{RESULTS}stats/']


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def receptive_field(n_blocks, kernel=3, convs_per_block=2):
    return 1 + sum(convs_per_block * (kernel - 1) * 2 ** i for i in range(n_blocks))

In [12]:
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, dilation=1, dropout=0.2):
        super().__init__()
        pad = (kernel - 1) * dilation
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, dilation=dilation, padding=pad),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel, dilation=dilation, padding=pad),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        out = out[:, :, :x.size(2)]          # trim to original length
        return nn.functional.relu(out + self.downsample(x))

In [ ]:
class TCN_MHSA(nn.Module):
    def __init__(self, input_dim, num_classes,
                 tcn_channels, d_model, n_heads, dropout=0.2):
        super().__init__()

        # TCN stack
        layers, in_ch = [], input_dim
        for i, out_ch in enumerate(tcn_channels):
            layers.append(TCNBlock(in_ch, out_ch, dilation=2**i, dropout=dropout))
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)

        # Project to d_model for attention
        self.proj = nn.Linear(tcn_channels[-1], d_model)

        # Multi-Head Self-Attention
        self.attn    = nn.MultiheadAttention(d_model, n_heads,
                                             dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        # Classification head (for training TCN)
        self.cls_head  = nn.Linear(d_model, num_classes)

    def forward(self, x, return_features=False):
        # x: (B, W, F) → TCN expects (B, F, W)
        x = x.permute(0, 2, 1)
        x = self.tcn(x)
        x = x.permute(0, 2, 1)              # back to (B, W, C)

        x = self.proj(x)                    # (B, W, d_model)

        # Self-Attention
        attn_out, _ = self.attn(x, x, x)
        x = self.norm(x + self.dropout(attn_out))

        x = x.mean(dim=1)                   # global average pool → (B, d_model)

        if return_features:
            return x                        # pooled embedding shared by both heads

        return self.cls_head(x)             # for training

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha                   # per-class weight tensor

    def forward(self, logits, targets):
        ce   = nn.functional.cross_entropy(logits, targets,
                                           weight=self.alpha, reduction='none')
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()


def make_criterion(y):
    counts  = np.bincount(y, minlength=NUM_CLASSES)
    weights = np.where(counts > 1, 1.0 / np.maximum(counts, 1), 0.0)
    weights = weights / weights.sum() * NUM_CLASSES
    return FocalLoss(gamma=2.0, alpha=torch.tensor(weights, dtype=torch.float32).to(DEVICE))

In [ ]:
def tensor_loader(X, y, batch, shuffle=False, seed=0):
    data = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    gen = torch.Generator().manual_seed(seed)
    return DataLoader(data, batch_size=batch, shuffle=shuffle, generator=gen)


def train_stage1(model, train_loader, es_loader, criterion, params, verbose=False):
    optimizer = optim.Adam(model.parameters(), lr=params['lr'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    best_val, patience_count = float('inf'), 0
    history = {'train_losses': [], 'val_losses': [], 'train_accs': [], 'val_accs': []}

    for epoch in range(params['epochs']):
        model.train()
        tr_loss, tr_correct, tr_total = 0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            tr_loss    += loss.item()
            tr_correct += (out.argmax(1) == yb).sum().item()
            tr_total   += yb.size(0)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for xb, yb in es_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                val_loss    += criterion(out, yb).item()
                val_correct += (out.argmax(1) == yb).sum().item()
                val_total   += yb.size(0)

        tr_loss  /= len(train_loader)
        val_loss /= len(es_loader)
        history['train_losses'].append(tr_loss)
        history['val_losses'].append(val_loss)
        history['train_accs'].append(tr_correct / tr_total * 100)
        history['val_accs'].append(val_correct / val_total * 100)

        scheduler.step(val_loss)
        if verbose:
            print(f"Epoch {epoch+1:02d}/{params['epochs']} | Train Loss: {tr_loss:.4f} | ES Loss: {val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), f'{SAVE_DIR}best_tcn.pt')
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= params['patience']:
                break

    model.load_state_dict(torch.load(f'{SAVE_DIR}best_tcn.pt'))
    model.eval()
    return history


def embed_arrays(model, X, batch=1024):
    with torch.no_grad():
        return np.concatenate([model(torch.tensor(X[i:i + batch], dtype=torch.float32).to(DEVICE),
                                     return_features=True).cpu().numpy()
                               for i in range(0, len(X), batch)])


def embed_starts(model, starts, W, art):
    with torch.no_grad():
        return np.concatenate([model(torch.tensor(x).to(DEVICE), return_features=True).cpu().numpy()
                               for x in load_windows(starts, W, art)])


def fit_lgbm(X_tr, y_tr, X_es, y_es, params, seed):
    seen = np.isin(y_es, np.unique(y_tr))
    head = lgb.LGBMClassifier(objective='multiclass', metric='multi_logloss', n_estimators=500,
                              learning_rate=params['lgb_lr'], num_leaves=params['num_leaves'],
                              n_jobs=-1, random_state=seed, verbose=-1)
    head.fit(X_tr, y_tr, eval_set=[(X_es[seen], y_es[seen])],
             callbacks=[lgb.early_stopping(params['lgb_rounds'], verbose=False)])
    return head


def fit_softmax(F_tr, y_tr, F_es, y_es, params, seed):
    set_seed(seed)
    head = nn.Linear(F_tr.shape[1], NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(head.parameters(), lr=params['lr'])
    train_loader = tensor_loader(F_tr, y_tr, params['batch'], shuffle=True, seed=seed)
    F_es_t = torch.tensor(F_es, dtype=torch.float32).to(DEVICE)
    y_es_t = torch.tensor(y_es, dtype=torch.long).to(DEVICE)
    best, best_state, wait = float('inf'), None, 0
    for _ in range(params['epochs']):
        for xb, yb in train_loader:
            optimizer.zero_grad()
            nn.functional.cross_entropy(head(xb.to(DEVICE)), yb.to(DEVICE)).backward()
            optimizer.step()
        with torch.no_grad():
            loss = nn.functional.cross_entropy(head(F_es_t), y_es_t).item()
        if loss < best:
            best, best_state, wait = loss, {k: v.clone() for k, v in head.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= params['patience']:
                break
    head.load_state_dict(best_state)
    return head.eval()


def softmax_predict(head, F, batch=65536):
    with torch.no_grad():
        return np.concatenate([head(torch.tensor(F[i:i + batch], dtype=torch.float32).to(DEVICE))
                               .argmax(1).cpu().numpy() for i in range(0, len(F), batch)])


def weighted_metrics(y, pred):
    acc = accuracy_score(y, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y, pred, average='weighted', zero_division=0)
    assert abs(acc - rec) < 1e-9
    return {'accuracy': acc * 100, 'precision_w': prec * 100, 'recall_w': rec * 100, 'f1_w': f1 * 100}


def fold_tensors(k, seed, W, S, config):
    art = fit_fold(k)
    with stage_timer('sequence_preparation', k, seed, config):
        X_tr, y_tr, X_es, y_es = training_windows(k, seed, W, S, art)
        X_bal, y_bal = balance(X_tr, y_tr, seed, k, config)
    return art, X_bal, y_bal, X_es, y_es


def result_row(method, config, head, W, S, seed, k, metrics):
    return {'dataset': DATASET, 'method': method, 'config': config, 'head': head,
            'W': W, 'S': S, 'seed': seed, 'fold': k, **metrics}


def run_tcal(k, seed, params=TABLE_I, W=WINDOW, S=STEP, config='full', controls=True, verbose=False):
    art, X_bal, y_bal, X_es, y_es = fold_tensors(k, seed, W, S, config)
    val_starts = block_starts(k, W, S)
    y_val = LABELS[val_starts + W - 1]

    set_seed(seed)
    model = TCN_MHSA(len(art['fidx']), NUM_CLASSES, params['channels'],
                     params['d_model'], params['n_heads']).to(DEVICE)
    train_loader = tensor_loader(X_bal, y_bal, params['batch'], shuffle=True, seed=seed)
    es_loader = tensor_loader(X_es, y_es, params['batch'])
    with stage_timer('stage1_tcn_mhsa', k, seed, config):
        history = train_stage1(model, train_loader, es_loader, make_criterion(y_bal), params, verbose)

    with stage_timer('deep_feature_extraction', k, seed, config):
        F_tr, F_es = embed_arrays(model, X_bal), embed_arrays(model, X_es)
        F_val = embed_starts(model, val_starts, W, art)

    with stage_timer('stage2_lightgbm', k, seed, config):
        lgb_model = fit_lgbm(F_tr, y_bal, F_es, y_es, params, seed)
    y_pred = lgb_model.predict(F_val)
    rows = [result_row('TCAL-IDS', config, 'lgbm', W, S, seed, k, weighted_metrics(y_val, y_pred))]

    if controls:
        softmax = fit_softmax(F_tr, y_bal, F_es, y_es, params, seed)
        rows.append(result_row('TCAL-IDS', config, 'softmax', W, S, seed, k,
                               weighted_metrics(y_val, softmax_predict(softmax, F_val))))

        flat = fit_lgbm(X_bal.reshape(len(X_bal), -1), y_bal, X_es.reshape(len(X_es), -1), y_es, params, seed)
        flat_pred = np.concatenate([flat.predict(x.reshape(len(x), -1)) for x in load_windows(val_starts, W, art)])
        rows.append(result_row('TCAL-IDS', 'lgbm_only', 'lgbm', W, S, seed, k, weighted_metrics(y_val, flat_pred)))

    last = {'model': model, 'lgb_model': lgb_model, 'L_val': y_val, 'y_pred': y_pred, 'art': art, **history}
    return rows, last


def run_baseline(name, k, seed, params, W=WINDOW, S=STEP):
    art, X_bal, y_bal, X_es, y_es = fold_tensors(k, seed, W, S, name)
    val_starts = block_starts(k, W, S)
    y_val = LABELS[val_starts + W - 1]
    set_seed(seed)
    model = BASELINES[name]['builder'](len(art['fidx']), NUM_CLASSES, **params).to(DEVICE)
    train_stage1(model, tensor_loader(X_bal, y_bal, TABLE_I['batch'], shuffle=True, seed=seed),
                 tensor_loader(X_es, y_es, TABLE_I['batch']), make_criterion(y_bal),
                 {**TABLE_I, 'lr': params.get('lr', TABLE_I['lr'])})
    with torch.no_grad():
        pred = np.concatenate([model(torch.tensor(x).to(DEVICE)).argmax(1).cpu().numpy()
                               for x in load_windows(val_starts, W, art)])
    return [result_row(name, 'full', 'softmax', W, S, seed, k, weighted_metrics(y_val, pred))]

In [ ]:
# builder(input_dim, num_classes, **params) -> nn.Module returning logits for (B, W, F).
BASELINES = {name: {'builder': None, 'space': {}} for name in
             ['Packet-BERT', 'Seq2Seq ConvLSTM', 'Pre Dyn-Mamba', 'Extractace-DNN', 'Diff-IDS']}

SEARCH_SPACE_TCAL = {
    'channels': [[32, 64], [64, 128], [32, 64, 128]],
    'd_model': [32, 64, 128], 'n_heads': [2, 4, 8], 'lr': [3e-4, 1e-3, 3e-3],
    'num_leaves': [31, 63, 127], 'lgb_lr': [0.01, 0.05, 0.1],
}


def random_search(method, space, n_trials=N_TRIALS):
    path = f'{RESULTS}search/{method}_{DATASET}.jsonl'
    done = sum(1 for _ in open(path)) if os.path.exists(path) else 0
    rng = random.Random(f'{method}-{DATASET}')
    for trial in range(n_trials):
        drawn = {name: rng.choice(values) for name, values in space.items()}
        if trial < done or out_of_time():
            continue
        scores = []
        for k in range(K):
            if method == 'TCAL-IDS':
                rows, _ = run_tcal(k, SEEDS[0], {**TABLE_I, **drawn}, config='search', controls=False)
            else:
                rows = run_baseline(method, k, SEEDS[0], drawn)
            scores.append(rows[0]['f1_w'])
        with open(path, 'a') as f:
            f.write(json.dumps({'trial': trial, 'params': drawn, 'f1_w': float(np.mean(scores))}) + '\n')
    trials = [json.loads(line) for line in open(path)]
    return max(trials, key=lambda t: t['f1_w'])


def load_csvs(pattern):
    files = glob.glob(pattern)
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()


RUNS_CSV = f'{RESULTS}runs/{DATASET}.csv'


def completed(method, config, W=WINDOW, S=STEP, path=RUNS_CSV):
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path)
    df = df[(df['method'] == method) & (df['config'] == config) & (df['W'] == W) & (df['S'] == S)]
    return set(zip(df['seed'], df['fold']))

In [ ]:
PARAMS = dict(TABLE_I)
if RUN_SEARCH:
    best = random_search('TCAL-IDS', SEARCH_SPACE_TCAL)
    print('Best TCAL-IDS trial:', best)
    if USE_SEARCH_BEST:
        PARAMS.update(best['params'])
for name, spec in BASELINES.items():
    if spec['builder'] is not None and (RUN_SEARCH or os.path.exists(f'{RESULTS}search/{name}_{DATASET}.jsonl')):
        spec['best'] = random_search(name, spec['space'], N_TRIALS if RUN_SEARCH else 0)['params']

params_path = f'{RESULTS}params.json'
if os.path.exists(params_path):
    assert json.load(open(params_path)) == PARAMS, 'results/ holds runs made with different settings'
else:
    json.dump(PARAMS, open(params_path, 'w'))

rf =receptive_field(len(PARAMS['channels']))
print(f'Receptive field r = {rf} | W = {WINDOW}')
if rf < WINDOW:
    print(f'WARNING: W = {WINDOW} exceeds the receptive field r = {rf}')

last = None
if RUN_MAIN:
    for seed in SEEDS:
        for k in range(K):
            if out_of_time():
                continue
            if (seed, k) not in completed('TCAL-IDS', 'full'):
                rows, last = run_tcal(k, seed, PARAMS, verbose=(seed == SEEDS[0] and k == 0))
                append_csv(RUNS_CSV, rows)
            for name, spec in BASELINES.items():
                if spec['builder'] is not None and (seed, k) not in completed(name, 'full'):
                    append_csv(RUNS_CSV, run_baseline(name, k, seed, spec.get('best', {})))
            gc.collect()
            torch.cuda.empty_cache()
    if out_of_time():
        print('Time limit reached: save results/ and set RESUME_FROM to continue.')

if last is not None:
    model, lgb_model = last['model'], last['lgb_model']
    L_val, y_pred = last['L_val'], last['y_pred']
    train_losses, val_losses = last['train_losses'], last['val_losses']
    train_accs, val_accs = last['train_accs'], last['val_accs']

In [ ]:
METRICS = ['accuracy', 'precision_w', 'recall_w', 'f1_w']


def fold_summary(df):
    rows = []
    for (method, config, head), group in df.groupby(['method', 'config', 'head']):
        folds = group.groupby('fold')[METRICS].mean()
        n = len(folds)
        if n < 2:
            continue
        for metric in METRICS:
            mean, sd = folds[metric].mean(), folds[metric].std(ddof=1)
            half = scipy_stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
            rows.append({'method': method, 'config': config, 'head': head, 'metric': metric,
                         'mean': mean, 'sd': sd, 'ci_low': mean - half, 'ci_high': mean + half,
                         **{f'fold_{f}': v for f, v in folds[metric].items()}})
    return pd.DataFrame(rows)


runs = load_csvs(f'{RESULTS}runs/*.csv')
if len(runs):
    summary = fold_summary(runs[runs['dataset'] == DATASET])
    summary.to_csv(f'{RESULTS}summary/{DATASET}_fold_summary.csv', index=False)
    print(summary[['method', 'config', 'head', 'metric', 'mean', 'sd', 'ci_low', 'ci_high']].round(4))

    full = runs[(runs['dataset'] == DATASET) & (runs['method'] == 'TCAL-IDS')
                & (runs['config'] == 'full') & (runs['head'] == 'lgbm')]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, metric in zip(axes, METRICS):
        ax.boxplot([full[full['fold'] == k][metric].values for k in range(K)])
        ax.set_xticklabels([f'F{k + 1}' for k in range(K)])
        ax.set_title(f'{DATASET} - {metric}')
        ax.set_ylabel('%')
    plt.tight_layout()
    plt.savefig(f'{RESULTS}summary/{DATASET}_fold_boxplots.pdf')
    plt.show()

In [ ]:
def holm(raw, alpha=0.05):
    raw = raw.sort_values('p_raw').reset_index(drop=True)
    m = len(raw)
    raw['rank'] = np.arange(1, m + 1)
    raw['threshold'] = alpha / (m - raw.index)
    raw['p_holm'] = np.minimum(1.0, (m - raw.index) * raw['p_raw']).cummax()
    raw['significant'] = raw['p_holm'] < alpha
    return raw


def paired_comparisons(df, reference='TCAL-IDS'):
    selected = df[(df['config'] == 'full')
                  & (((df['method'] == reference) & (df['head'] == 'lgbm')) | (df['method'] != reference))]
    per_seed = selected.groupby(['method', 'seed'])['accuracy'].mean().unstack('method')
    rows = []
    for method in [c for c in per_seed.columns if c != reference]:
        pair = per_seed[[reference, method]].dropna()
        diff = pair[reference] - pair[method]
        t, p = scipy_stats.ttest_rel(pair[reference], pair[method])
        rows.append({'dataset': DATASET, 'comparison': f'{reference} vs {method}', 'n_runs': len(diff),
                     'mean_diff': diff.mean(), 'sd_diff': diff.std(ddof=1),
                     'se': diff.std(ddof=1) / np.sqrt(len(diff)), 't': t,
                     'dz': diff.mean() / diff.std(ddof=1), 'p_raw': p})
    return pd.DataFrame(rows)


if len(runs):
    comparisons = paired_comparisons(runs[runs['dataset'] == DATASET])
    if len(comparisons):
        comparisons.to_csv(f'{RESULTS}stats/raw_p_{DATASET}.csv', index=False)
    family = pd.concat([load_csvs(f'{d}raw_p_*.csv') for d in STATS_DIRS], ignore_index=True)
    if len(family):
        ladder = holm(family)
        ladder.to_csv(f'{RESULTS}stats/holm_ladder.csv', index=False)
        print(f'Holm family m = {len(ladder)} (15 expected once all datasets and baselines are present)')
        print(ladder[['rank', 'dataset', 'comparison', 'p_raw', 'threshold', 'p_holm', 'dz']])
    else:
        print('No baseline runs found: paired tests and Holm ladder skipped.')

In [ ]:
if RUN_SWEEP:
    sweep_csv = f'{RESULTS}sensitivity/ws_{DATASET}.csv'
    settings = [(w, STEP) for w in SWEEP_W] + [(WINDOW, s) for s in SWEEP_S if s != STEP]
    for W, S in settings:
        if receptive_field(len(PARAMS['channels'])) < W:
            print(f'W = {W} exceeds r = {receptive_field(len(PARAMS["channels"]))}')
        for seed in SWEEP_SEEDS:
            for k in range(K):
                if out_of_time() or (seed, k) in completed('TCAL-IDS', 'sweep', W, S, sweep_csv):
                    continue
                rows, _ = run_tcal(k, seed, PARAMS, W, S, config='sweep', controls=False)
                append_csv(sweep_csv, rows)
    sweep = pd.read_csv(sweep_csv)
    per_fold = sweep.groupby(['W', 'S', 'fold'])[METRICS].mean()
    print(per_fold.groupby(['W', 'S']).agg(['mean', lambda v: v.std(ddof=1)]).round(4))

In [ ]:
if RUN_PROFILE and last is not None:
    F_dim = len(last['art']['fidx'])
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    probe = torch.zeros(1, WINDOW, F_dim, device=DEVICE)
    with FlopCounterMode(model, display=False) as counter:
        with torch.no_grad():
            model(probe, return_features=True)
    timings = []
    for _ in range(100):
        start = time.perf_counter()
        with torch.no_grad():
            z = model(probe, return_features=True).cpu().numpy()
        lgb_model.predict(z)
        timings.append(time.perf_counter() - start)
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    profile = {'dataset': DATASET, 'device': device_name, 'params_trainable': n_params,
               'encoder_gflops': counter.get_total_flops() / 1e9,
               'tree_count': lgb_model.booster_.num_trees(),
               'latency_s_per_sequence': float(np.median(timings))}
    append_csv(f'{RESULTS}profile/{DATASET}.csv', [profile])
    print(profile)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

if last is not None:
    all_labels = np.arange(NUM_CLASSES)
    print(classification_report(L_val, y_pred, labels=all_labels, zero_division=0,
          target_names=[str(c) for c in le.classes_]))

    cm = confusion_matrix(L_val, y_pred, labels=all_labels)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'{DATASET} — Confusion Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
if last is not None:
    epochs_ran = range(1, len(train_losses) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    ax1.plot(epochs_ran, train_losses, label='Train Loss', marker='o', markersize=3)
    ax1.plot(epochs_ran, val_losses,   label='Early-stop Loss', marker='o', markersize=3)
    ax1.set_title(f'{DATASET} — Loss vs Epoch')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy
    ax2.plot(epochs_ran, train_accs, label='Train Acc', marker='o', markersize=3)
    ax2.plot(epochs_ran, val_accs,   label='Early-stop Acc', marker='o', markersize=3)
    ax2.set_title(f'{DATASET} — Accuracy vs Epoch')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}training_curves.png', dpi=150)
    plt.show()

In [ ]:
import joblib

if last is not None:
    torch.save(model.state_dict(), f'{SAVE_DIR}best_tcn.pt')
    joblib.dump(lgb_model,         f'{SAVE_DIR}lgbm.pkl')
    joblib.dump(le,                f'{SAVE_DIR}label_encoder.pkl')

    print(f"Saved → {SAVE_DIR}")